# 🏭 Smart Warehouse Inventory Optimizer
### Exploratory Data Analysis (EDA)
> **Pragyan AI Hackathon** | Demand Forecasting & Inventory Intelligence

---
**Dataset:** `inventory_demand_forecasting_dataset.csv`  
**Covers:** 3 Stores · 4 Products · 365 Days · 4,380 Records

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print('✅ Libraries loaded successfully')

## 📂 2. Load & Inspect Dataset

In [ ]:
df = pd.read_csv('../data/inventory_demand_forecasting_dataset.csv')
df['date'] = pd.to_datetime(df['date'])

print('Shape         :', df.shape)
print('Date Range    :', df['date'].min().date(), '→', df['date'].max().date())
print('Stores        :', df['store_id'].unique().tolist())
print('Products      :', df['product_id'].unique().tolist())
print('Missing Values:\n', df.isnull().sum())
df.head(10)

In [ ]:
df.describe().round(2)

## 📊 3. Demand Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram
axes[0].hist(df['demand'], bins=30, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].axvline(df['demand'].mean(), color='red', linestyle='--', label=f'Mean: {df["demand"].mean():.1f}')
axes[0].axvline(df['demand'].median(), color='orange', linestyle='--', label=f'Median: {df["demand"].median():.1f}')
axes[0].set_title('Demand Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Demand (Units)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Boxplot per product
df.boxplot(column='demand', by='product_id', ax=axes[1],
           boxprops=dict(color='#3498db'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Demand by Product', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Product')
axes[1].set_ylabel('Demand (Units)')
plt.suptitle('')
plt.tight_layout()
plt.show()

print(f'Skewness: {df["demand"].skew():.3f} | Kurtosis: {df["demand"].kurt():.3f}')

## 📈 4. Monthly Demand Trend

In [ ]:
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')
df['week'] = df['date'].dt.isocalendar().week

monthly = df.groupby(['month', 'month_name', 'product_id'])['demand'].mean().reset_index()

plt.figure(figsize=(13, 5))
colors = {'Product_A': '#00D4FF', 'Product_B': '#7B61FF',
          'Product_C': '#00FF9C', 'Product_D': '#FF6B6B'}

for pid, color in colors.items():
    pdata = monthly[monthly['product_id'] == pid].sort_values('month')
    plt.plot(pdata['month_name'], pdata['demand'], marker='o',
             linewidth=2.5, color=color, label=pid)
    plt.fill_between(range(len(pdata)), pdata['demand'], alpha=0.07, color=color)

plt.title('📈 Monthly Avg Demand per Product — 2024', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Avg Daily Demand (Units)')
plt.legend()
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## 🏪 5. Store-Level Analysis

In [ ]:
store_summary = df.groupby('store_id').agg(
    avg_demand   = ('demand',    'mean'),
    total_demand = ('demand',    'sum'),
    avg_price    = ('price',     'mean'),
    promo_rate   = ('promotion', 'mean'),
).round(2)
store_summary['promo_rate'] = (store_summary['promo_rate'] * 100).round(1)
print(store_summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Avg demand per store
store_summary['avg_demand'].plot(kind='bar', ax=axes[0],
    color=['#00D4FF','#7B61FF','#00FF9C'], edgecolor='white', width=0.5)
axes[0].set_title('Avg Demand by Store', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Avg Daily Demand')
axes[0].tick_params(axis='x', rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{bar.get_height():.1f}', ha='center', fontsize=10, fontweight='bold')

# Store x Product heatmap
pivot = df.groupby(['store_id','product_id'])['demand'].mean().unstack()
sns.heatmap(pivot, ax=axes[1], annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Avg Demand'})
axes[1].set_title('Demand Heatmap: Store × Product', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 🎯 6. Promotion Impact Analysis

In [ ]:
promo = df.groupby(['product_id','promotion'])['demand'].mean().unstack()
promo.columns = ['No Promo', 'With Promo']
promo['Uplift %'] = ((promo['With Promo'] - promo['No Promo']) / promo['No Promo'] * 100).round(2)
print(promo.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(promo))
axes[0].bar(x - 0.2, promo['No Promo'],   0.35, label='No Promotion', color='#95a5a6', alpha=0.85)
axes[0].bar(x + 0.2, promo['With Promo'], 0.35, label='With Promotion', color='#00FF9C', alpha=0.9)
axes[0].set_xticks(x)
axes[0].set_xticklabels(promo.index)
axes[0].set_title('Promo vs No-Promo Demand', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Avg Demand (Units)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.4)

bars = axes[1].bar(promo.index, promo['Uplift %'],
                    color=['#00D4FF','#7B61FF','#00FF9C','#FF6B6B'], alpha=0.9)
for bar, val in zip(bars, promo['Uplift %']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'+{val:.1f}%', ha='center', fontsize=11, fontweight='bold', color='green')
axes[1].set_title('Promotion Uplift % per Product', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Uplift (%)')
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

## 💲 7. Price Sensitivity Analysis

In [ ]:
corr = df[['price','demand','promotion']].corr().round(3)
print('Correlation Matrix:\n', corr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter per product with trendline
colors = {'Product_A':'#00D4FF','Product_B':'#7B61FF',
          'Product_C':'#00FF9C','Product_D':'#FF6B6B'}
for pid, color in colors.items():
    pdata = df[df['product_id'] == pid]
    axes[0].scatter(pdata['price'], pdata['demand'], alpha=0.2, s=12, color=color, label=pid)
    z = np.polyfit(pdata['price'], pdata['demand'], 1)
    xs = np.linspace(pdata['price'].min(), pdata['price'].max(), 100)
    axes[0].plot(xs, np.poly1d(z)(xs), color=color, linewidth=2)
axes[0].set_title('Price vs Demand (with Trend)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Demand (Units)')
axes[0].legend()

# Correlation heatmap
sns.heatmap(corr, ax=axes[1], annot=True, fmt='.3f',
            cmap='coolwarm', center=0, linewidths=1,
            cbar_kws={'shrink': 0.8})
axes[1].set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 🧮 8. Inventory Metrics: Safety Stock & Reorder Point

In [ ]:
# Formula constants
Z = 1.645          # 95% service level
LEAD_TIME = 7      # days
ORDER_COST = 50    # $ per order
HOLDING_RATE = 0.2 # 20% of unit cost

product_stats = df.groupby('product_id').agg(
    avg_demand = ('demand', 'mean'),
    std_demand = ('demand', 'std'),
    total_demand = ('demand', 'sum'),
    avg_price  = ('price',  'mean'),
).round(3)

product_stats['safety_stock'] = (
    Z * product_stats['std_demand'] * np.sqrt(LEAD_TIME)
).round(1)

product_stats['reorder_point'] = (
    product_stats['avg_demand'] * LEAD_TIME + product_stats['safety_stock']
).round(1)

product_stats['EOQ'] = np.sqrt(
    2 * product_stats['total_demand'] * ORDER_COST /
    (HOLDING_RATE * product_stats['avg_price'])
).round(1)

print('📦 Inventory Intelligence Metrics')
print('='*60)
print(product_stats[['avg_demand','std_demand','safety_stock','reorder_point','EOQ']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(product_stats))
axes[0].bar(x - 0.2, product_stats['safety_stock'],  0.35,
            label='Safety Stock', color='#FFB347', alpha=0.9)
axes[0].bar(x + 0.2, product_stats['reorder_point'], 0.35,
            label='Reorder Point', color='#FF6B6B', alpha=0.9)
axes[0].set_xticks(x)
axes[0].set_xticklabels(product_stats.index)
axes[0].set_title('Safety Stock vs Reorder Point', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Units')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.4)

axes[1].bar(product_stats.index, product_stats['EOQ'],
            color=['#00D4FF','#7B61FF','#00FF9C','#FF6B6B'], alpha=0.9)
axes[1].set_title('Economic Order Quantity (EOQ)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Units per Order')
axes[1].grid(axis='y', alpha=0.4)
for bar in axes[1].patches:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{bar.get_height():.0f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 🗓️ 9. Weekly Demand Pattern

In [ ]:
df['dayofweek'] = df['date'].dt.day_name()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekly = df.groupby('dayofweek')['demand'].mean().reindex(day_order)

plt.figure(figsize=(11, 4))
bars = plt.bar(weekly.index, weekly.values,
               color=['#00D4FF' if v == weekly.max() else '#3d5a80' for v in weekly.values],
               edgecolor='white', alpha=0.9)
plt.axhline(weekly.mean(), color='orange', linestyle='--',
            linewidth=1.5, label=f'Avg: {weekly.mean():.1f}')
for bar in bars:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'{bar.get_height():.1f}', ha='center', fontsize=9, fontweight='bold')
plt.title('🗓️ Avg Demand by Day of Week', fontsize=14, fontweight='bold')
plt.ylabel('Avg Demand (Units)')
plt.legend()
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## 🔍 10. Key Findings Summary

In [ ]:
promo_avg    = df[df['promotion']==1]['demand'].mean()
no_promo_avg = df[df['promotion']==0]['demand'].mean()
uplift       = (promo_avg - no_promo_avg) / no_promo_avg * 100
corr_val     = df['price'].corr(df['demand'])
peak_m       = df.groupby(df['date'].dt.month)['demand'].mean().idxmax()

findings = {
    '📦 Total Records'         : f"{len(df):,}",
    '🏪 Stores Covered'        : df['store_id'].nunique(),
    '🛒 Products Covered'      : df['product_id'].nunique(),
    '📊 Avg Daily Demand'      : f"{df['demand'].mean():.2f} units",
    '🎯 Promo Demand Uplift'   : f"+{uplift:.1f}%",
    '💲 Price-Demand Corr'     : f"{corr_val:.3f}",
    '📅 Peak Demand Month'     : pd.Timestamp(2024, peak_m, 1).strftime('%B'),
    '✅ Missing Values'        : "None (Clean)",
    '🔁 Promotion Coverage'    : f"{df['promotion'].mean()*100:.1f}% of records",
}

print('='*45)
print('   🏭 EDA Summary — Inventory Optimizer')
print('='*45)
for k, v in findings.items():
    print(f'  {k:<28} {v}')
print('='*45)
print('✅ EDA Complete — Ready for Dashboard & Modeling')